# Document Parsing using Unlimited-OCR and OpenVINO

**[Unlimited-OCR](https://huggingface.co/baidu/Unlimited-OCR)** is a vision-language model (VLM) from Baidu for efficient, high-resolution document understanding and optical character recognition (OCR). It pairs a deep dual vision encoder (SAM ViT-B + CLIP-L) and a linear projector with a compact DeepSeek-V2 **Mixture-of-Experts** decoder (12 layers, 64 routed + 2 shared experts, top-6). Every decoder layer uses **sliding-window attention** (window = 128) during generation while keeping all prefill (image/prompt) tokens visible, and dynamic image tiling lets it read very large documents.

More details can be found in the original [model card](https://huggingface.co/baidu/Unlimited-OCR).

In this tutorial we consider how to convert and run Unlimited-OCR using [OpenVINO](https://github.com/openvinotoolkit/openvino) and optimize it using [NNCF](https://github.com/openvinotoolkit/nncf).

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download the original model](#Download-the-original-model)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
- [Prepare inference pipeline](#Prepare-inference-pipeline)
- [Run model inference](#Run-model-inference)
- [Interactive demo](#Interactive-demo)

### Installation instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start. For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/unlimited-ocr/unlimited-ocr.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import requests
from pathlib import Path

utility_files = ["cmd_helper.py", "notebook_utils.py", "pip_helper.py"]
base_utility_url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/"

for utility in utility_files:
    if not Path(utility).exists():
        r = requests.get(base_utility_url + utility)
        with open(utility, "w", encoding="utf-8") as f:
            f.write(r.text)

In [ ]:
from pip_helper import pip_install

pip_install("-Uq", "--pre", "openvino", "--extra-index-url", "https://storage.openvinotoolkit.org/simple/wheels/nightly")
pip_install(
    "-q",
    "nncf>=2.15",
    "torch==2.8.0",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "torchvision==0.23.0",
    "einops",
    "addict",
    "easydict",
    "huggingface_hub",
    "accelerate>=0.26.0",
    "PyMuPDF",
    "gradio>=4.19,<6",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)

## Download the original model
[back to top ⬆️](#Table-of-contents:)

The model is downloaded from the Hugging Face Hub into a local folder named `Unlimited_OCR` (with an underscore so it can be imported as a Python package by the helper).

In [ ]:
from huggingface_hub import snapshot_download

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("unlimited-ocr.ipynb")

model_id = "baidu/Unlimited-OCR"
local_dir = "Unlimited_OCR"

print(f"Downloading {model_id} to {local_dir}...")
snapshot_download(repo_id=model_id, local_dir=local_dir, local_dir_use_symlinks=False)
print(f"Model downloaded successfully to {local_dir}")

## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)

Unlimited-OCR is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). The [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html) should be used for these purposes. `ov.convert_model` accepts the original PyTorch model instance and example input for tracing and returns an `ov.Model`. The converted model can be saved with `ov.save_model` or loaded directly with `core.compile_model`.

The script `ov_unlimited_ocr_helper.py` contains the helper function `convert_unlimited_ocr` for model conversion. It splits the model into the following OpenVINO sub-models:

- **text embeddings** — token embedding lookup;
- **vision embeddings** — the SAM + CLIP encoders and the linear projector, exported at two fixed square resolutions (the 1024 global view and the 640 crop tiles), each with a dynamic batch dimension so the variable number of crop tiles is handled at runtime. Two IRs are used because the SAM/CLIP positional-embedding interpolation is resolution dependent and cannot be traced as a single dynamic-resolution graph;
- **language model** — the stateful DeepSeek-V2 MoE decoder. Its MoE routing is rewritten as a static loop over all experts, and the sliding-window attention (window = 128) is reproduced with an explicit additive mask built at inference time so every query attends to all prefill tokens plus the last 128 generated tokens.

Please check the helper's content for conversion details.

Weight compression with [NNCF](https://github.com/openvinotoolkit/nncf) can be optionally applied to reduce the model footprint. When the checkbox below is enabled, the language model is compressed to INT4 (`INT4_SYM`, group size 64) and the vision encoder to INT8 (`INT8_ASYM`).

In [ ]:
import ipywidgets as widgets

to_compress = widgets.Checkbox(value=True, description="Compression")
to_compress

In [ ]:
import nncf
from ov_unlimited_ocr_helper import convert_unlimited_ocr

quantization_config = None
if to_compress.value:
    quantization_config = {
        "vision": {"mode": nncf.CompressWeightsMode.INT8_ASYM},
        "llm": {"mode": nncf.CompressWeightsMode.INT4_SYM, "group_size": 64, "ratio": 1.0},
    }

model_path = Path(local_dir) / ("INT4" if to_compress.value else "FP16")
convert_unlimited_ocr(local_dir, model_path=model_path, quantization_config=quantization_config)

## Prepare inference pipeline
[back to top ⬆️](#Table-of-contents:)

The `OVUnlimitedOCRForCausalLM` class defined in `ov_unlimited_ocr_helper.py` represents the model inference class. It accepts a path to the converted model directory and a target device for inference. Like the original pipeline, it exposes an `infer` method that takes a tokenizer, a prompt and an image file.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

In [ ]:
from transformers import AutoTokenizer
from ov_unlimited_ocr_helper import OVUnlimitedOCRForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = OVUnlimitedOCRForCausalLM(model_path, device=device.value)

## Run model inference
[back to top ⬆️](#Table-of-contents:)

Let's check the model prediction for document parsing.

In [ ]:
from PIL import Image

url = "https://huggingface.co/spaces/khang119966/DeepSeek-OCR-DEMO/resolve/main/doc_markdown.png"
file_name = "doc_markdown.png"
if not Path(file_name).exists():
    Image.open(requests.get(url, stream=True).raw).save(file_name)

Image.open(file_name)

In [ ]:
prompt = "<image>document parsing."
output_path = "."

res = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=file_name,
    output_path=output_path,
    base_size=1024,
    image_size=640,
    crop_mode=True,
    no_repeat_ngram_size=35,
    ngram_window=128,
    save_results=True,
    test_compress=True,
)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, tokenizer)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/